# Transformer Implementation in Pytorch


This is my implementation of the transformer architecture, explaining as we go through

Reference paper: https://arxiv.org/pdf/1706.03762


In [1]:
import os
import torch
import math
import numpy as np
import torch.nn as nn
import torch.nn.functional as F 

#Just import statements, math performs faster at scalar operations, numpy better for vectorized.

## Approach and Components
We will use another tokenizer:
    - tiktoken (General Purpose) 
    - mathBERT (Latex Math)
We will need to build:
1. Encoder
    1. Positional Encoding + Embedding
    2. Multi-head attention
        - Single Head attention with QKV
2. Decoder
    1. Positional Encoding + Embedding
    2. Masked Multi-Head attention
        - Single Head att.
    3. Multi Head att.
3. Linear + Softmax Ouput

Where all of these will have residual connections as well as a normalization after each block


Note we will actually implement some changes from the original model.
 - Layer Norm before sublayers, training performs better. 
 - Changing the Base 10000 for the positional encoding, we will be working with a different type of text.

In [2]:
# Setting the random seed for reproducibility
seed = 2103
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed) # For multi-GPU setups
    
# Configure CuDNN to use deterministic algorithms
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

Next we will make the Input Embedding along with the positional encoder. In the AIAYN paper positional encoding is done by using 512 different frequencys with varying orders of magnitude. In the paper they include this arbitrary seeming 10000 number for this computation, it is an empirically good number for capturing both relative relational position, as well as absolute.
The sinusoidal encoding itself is also good for extrapolating to lengths outside of our training context limit

## Positional Encoding Function

In [ ]:
@staticmethod
def create_positional_encoding(seq_length, d_model, posencode_basefreq=10000):
    # Create position indices: [seq_length, 1]
    position = torch.arange(seq_length, dtype=torch.float).unsqueeze(1)
    
    # Create dimension indices: [1, d_model//2]
    div_term = torch.exp(
        torch.arange(0, d_model, 2,dtype=torch.float) * 
        (-math.log(posencode_basefreq) / d_model)
    )
    
    # Create empty tensor: [seq_length, d_model]
    pe = torch.zeros(seq_length, d_model)
    
    # Compute sin and cos for even and odd dimensions
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    
    # dimension: [1, seq_length, d_model]
    pe = pe.unsqueeze(0)
    
    return pe

### Input Embedding

In [ ]:
    # testing dimensions
    # a = torch.rand(2, 10, 512)
    # print(a.shape[0])
    # tes = create_positional_encoding(a.shape[1], a.shape[2])
    # tes.size()

2


torch.Size([1, 10, 512])

In [ ]:
# Here we tokenize and embed our input.
class TokenEmbedding(nn.Module):
    def __init__(self, d_input, d_model, dropout=0.1, max_pos_embedding = 512):
        super().__init__(self)
        self.d_model = d_model
        self.layer1 = nn.Linear(d_input, d_model)
        self.dropout = nn.Dropout(dropout)
        self.embedding = nn.Embedding(max_pos_embedding, d_model)
    def forward(self, x):
        x2 = self.layer1(x)
        x2 = self.dropout(x2)
        x2 = self.embedding(torch.arange(x.size(1), device=x.device))
        return x2*math.sqrt(self.d_model) + create_positional_encoding(x,self.d_model)
        

## FeedForward Network
Here we implement one of the more simpler networks, two linear networks plus a layer norm and residual connection

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, dropout = 0.1 ):
        super().__init__(self)
        self.d_model = d_model
        self.layer1 = nn.Linear(4*d_model,d_model)
        self.layer2 = nn.Linear(d_model, 4*d_model)
        self.layernorm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x shape: []
        assert x.dtype == torch.long, "Input x must be of type Long"
        x2 = self.layer1(x)
        x2 = F.relu(x2)
        x2 = self.layernorm(x2)
        x2 = self.dropout(x2)
        # Note that we do a pre-norm here.
        return self.layer2(x2) + x


In [ ]:
@staticmethod
def scaled_dot_product_attention(q, k, v, mask=None):
    d_k = q.size(-1)
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e10)
    attn_weights = F.softmax(scores, dim=-1)
    output = torch.matmul(attn_weights, v)
    return output, attn_weights

In [ ]:
# @staticmethod
# def Attention(self, x):
#     # shape of x = [1, seq_length, d_model]
#     q = self.Q(x)
#     k = self.K(x)
#     v = self.V(x)
#     output, attn_weights = scaled_dot_product_attention(q, k, v)
#     output = self.dropout(output)
#     return output, attn_weights
# This is the implementation for single head attention, extending to multi-head.

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, h=4, dropout=0.1):
        super().__init__(self)
        self.d_model = d_model
        self.h = h
        self.d_k = d_model // h
        assert self.d_k * h == d_model, "d_model must be divisible by h"
        self.dropout = nn.Dropout(dropout)
        self.linears = nn.ModuleList([nn.Linear(d_model, d_model) for _ in range(4)])  # Q, K, V, and output linear layers 
        self.attn_weights = None  # To store attention weights for visualization

    def forward(self, q, k , v):
        # shape of x = [1, seq_length, d_model]

        q1, k1, v1 = [l_trans(x).view(x.size(0),-1,self.h,self.d_k) for l_trans, x  in zip(self.linears, (q,k,v))]
        output, attn_weights = scaled_dot_product_attention(q1, k1, v1)
        output = self.dropout(output)
        self.attn_weights = attn_weights 
        return output


In [ ]:
class MaskedMultiHeadAttention(nn.Module):
    def __init__(self, d_model, h=4, dropout=0.1, mask=None):
        super().__init__(self)
        self.d_model = d_model
        self.h = h
        self.d_k = d_model // h
        self.mask = mask
        assert self.d_k * h == d_model, "d_model must be divisible by h"
        self.dropout = nn.Dropout(dropout)
        self.linears = nn.ModuleList([nn.Linear(d_model, d_model) for _ in range(4)])  # Q, K, V, and output linear layers 
        self.attn_weights = None  # To store attention weights for visualization

    def forward(self, q, k , v):
        # shape of x = [1, seq_length, d_model]

        q1, k1, v1 = [l_trans(x).view(x.size(0),-1,self.h,self.d_k) for l_trans, x  in zip(self.linears, (q,k,v))]
        output, attn_weights = scaled_dot_product_attention(q1, k1, v1, mask=self.mask)
        output = self.dropout(output)
        self.attn_weights = attn_weights 
        return output


In [ ]:
class Decoder(nn.Module):
    def __init__(self, d_model, h, dropout=0.1):
        super().__init__(self)
        self.d_model = d_model
        self.h = h
        assert d_model % h == 0, "d_model must be divisible by h"
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        # shape of x = [1, seq_length, d_model]
        MaskedMultiHeadAttention()




In [ ]:
class Encoder(nn.Module):
    def __init__(self, d_model, h, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.h = h
        assert d_model % h == 0, "d_model must be divisible by h"
        self.dropout = nn.Dropout(dropout)
        self.layers = nn.ModuleList([[MultiHeadAttention(d_model, h, dropout), FeedForward(d_model, dropout)] for _ in range(6)]) 
    def forward(self, x):
        # shape of x = [1, seq_length, d_model]
        for attn, ff in self.layers:
            x = attn(x, x, x)
            x = ff(x)
        return x